# Rumor Detection — Colab 快速训练

本 Notebook 用于在 Google Colab 上快速训练谣言检测模型。

**建议**: 运行时选择 `代码执行程序` → `更改运行时类型` → `T4 GPU`

## 1. 检查 GPU 可用性

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')

## 2. 挂载 Google Drive（推荐，用于持久化保存模型）

如果不想用 Drive，可以跳过这步，训练完后直接下载 `checkpoints/` 文件夹即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 可选：设定 Drive 里的工作目录
import os
DRIVE_DIR = '/content/drive/MyDrive/rumor_detection'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive 工作目录:', DRIVE_DIR)

## 3. 拉取代码仓库

In [ ]:
# 替换为你的 GitHub 仓库地址
REPO_URL = 'https://github.com/EricQian06/Rumor-detection.git'

!git clone {REPO_URL}
# 注意：仓库名是大写的 Rumor-detection，Linux 区分大小写
%cd Rumor-detection
!ls -la

## 4. 安装依赖 + 设置 HuggingFace 镜像（可选）

如果下载模型权重较慢，可以先设置镜像：

In [ ]:
# 设置 HuggingFace 镜像（国内/下载慢时启用，默认注释掉）
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

!pip install -q -r requirements.txt

## 5. 上传数据文件（如果数据不在仓库中）

运行后会弹出文件选择器，请上传 `train.csv` 和 `val.csv`。
如果数据已经在仓库里，可跳过此步。

In [ ]:
from google.colab import files
uploaded = files.upload()  # 上传 train.csv 和 val.csv
print('已上传文件:', list(uploaded.keys()))

## 6. 一键训练

当前默认使用 **`microsoft/deberta-v3-base`**（推荐），它在分类任务上通常优于 `roberta-base`，且参数量相近，Colab T4 可稳定运行。

| 模型 | 参数量 | batch_size | 预期 val_acc |
|------|--------|-----------|-------------|
| `roberta-base` | 125M | 32 | ~87.8% (已有) |
| `microsoft/deberta-v3-base` | 86M | 32 | **+2~3%** |
| `roberta-large` | 355M | 8~16 | **+2~4%** |

> **防崩溃提示**：如果运行时断开，尝试把 `batch_size` 改为 `16`，或缩短 `max_len` 为 `128`。

### 方式 A：稳定脚本（推荐，避免 [object CloseEvent]）
直接在 Python 进程内运行，OOM 时不会断开连接：

修改 `colab_train_safe.py` 里的 `model_name` 即可切换模型。

### 方式 B：命令行训练（如果方式 A 也报错再尝试）

使用 deberta-v3-base：

In [ ]:
!python -m src.train \
    --train_csv train.csv \
    --val_csv val.csv \
    --model_name microsoft/deberta-v3-base \
    --epochs 5 \
    --batch_size 32 \
    --lr 2e-5 \
    --max_len 256 \
    --output_dir checkpoints \
    --num_workers 2

In [ ]:
!python -m src.train \
    --train_csv train.csv \
    --val_csv val.csv \
    --epochs 5 \
    --batch_size 32 \
    --lr 2e-5 \
    --max_len 256 \
    --output_dir checkpoints \
    --num_workers 2

In [ ]:
!python -m src.evaluate \
    --model_dir checkpoints \
    --val_csv val.csv \
    --model_name microsoft/deberta-v3-base

In [ ]:
!python -m src.evaluate \
    --model_dir checkpoints \
    --val_csv val.csv

## 8. 保存结果到 Google Drive（可选）

In [ ]:
import shutil

# 复制 checkpoints 和 results 到 Drive
shutil.copytree('checkpoints', f'{DRIVE_DIR}/checkpoints', dirs_exist_ok=True)
if os.path.exists('results'):
    shutil.copytree('results', f'{DRIVE_DIR}/results', dirs_exist_ok=True)
print('模型和结果已保存到 Drive:', DRIVE_DIR)

## 9. 打包下载（如果不使用 Drive）

In [ ]:
!zip -r checkpoints.zip checkpoints/
files.download('checkpoints.zip')

In [ ]:
# 示例：对比 roberta-base vs deberta-v3-base 在不同学习率下的表现
models = [
    ("roberta-base", 32),
    ("microsoft/deberta-v3-base", 32),
]
lrs = [1e-5, 2e-5]

for model_name, bs in models:
    for lr in lrs:
        out_dir = f'checkpoints_{model_name.replace("/", "_")}_lr{lr}'
        print(f'\n=== {model_name} | lr={lr} | bs={bs} ===')
        !python -m src.train \
            --train_csv train.csv \
            --val_csv val.csv \
            --model_name {model_name} \
            --epochs 3 \
            --batch_size {bs} \
            --lr {lr} \
            --max_len 256 \
            --output_dir {out_dir} \
            --num_workers 2

In [ ]:
# 示例：尝试不同的学习率
lrs = [1e-5, 2e-5, 3e-5]

for lr in lrs:
    out_dir = f'checkpoints_lr{lr}'
    print(f'\n=== Training with lr={lr} ===')
    !python -m src.train \
        --train_csv train.csv \
        --val_csv val.csv \
        --epochs 3 \
        --batch_size 32 \
        --lr {lr} \
        --max_len 256 \
        --output_dir {out_dir} \
        --num_workers 2